# Ablation Study: Phase × Embedding Dimension

Runs a 3×3 grid of experiments: `{phase 1, phase 2, both} × {D=128, 256, 512}`.

All training and evaluation code is self-contained — no subprocess calls. Set `DEBUG=True` for a fast smoke test.

## Configuration

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_ROOT    = "/content/data/dataset_a"       if IN_COLAB else "./datasets/dataset_a"
DATASET_ROOT = "/content/data/dataset_a"       if IN_COLAB else "./datasets/dataset_a"
SAVE_DIR     = "/content/checkpoints/ablation" if IN_COLAB else "./checkpoints/ablation"
RESULTS_DIR  = "/content/results/ablation"     if IN_COLAB else "./results/ablation"

BATCH_SIZE   = 128
NUM_WORKERS  = 0  # >0 causes multiprocessing errors in Jupyter on macOS
DEBUG        = False  # True: 500-image subset, 2 epochs per run

import torch
# Auto-detects: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

In [ ]:
import itertools
import math
import os
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from tqdm import tqdm

## Shared Model & Utilities

In [ ]:
class FaceEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Linear(2048, embedding_dim)
        self._embedding_dim = embedding_dim

    @property
    def embedding_dim(self):
        return self._embedding_dim

    def forward(self, x):
        return self.backbone(x)

    def encode(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)


def aggregate_template_features(embeddings, template_ids, media_ids):
    template_features = {}
    for tid in np.unique(template_ids):
        mask = template_ids == tid
        t_embs = embeddings[mask]
        t_mids = media_ids[mask]
        media_feats = []
        for mid in np.unique(t_mids):
            media_feats.append(t_embs[t_mids == mid].mean(axis=0))
        feat = np.sum(media_feats, axis=0)
        feat = feat / (np.linalg.norm(feat) + 1e-8)
        template_features[tid] = feat
    return template_features

## Training Functions

In [ ]:
class GridMask:
    def __init__(self, grid=4, drop_ratio=0.25, p=0.15):
        self.grid = grid
        self.n_drop = math.ceil(drop_ratio * grid * grid)
        self.p = p

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        _, H, W = tensor.shape
        cell_h = H // self.grid
        cell_w = W // self.grid
        cells = [(r, c) for r in range(self.grid) for c in range(self.grid)]
        drop = random.sample(cells, self.n_drop)
        tensor = tensor.clone()
        for r, c in drop:
            tensor[:, r * cell_h:(r + 1) * cell_h, c * cell_w:(c + 1) * cell_w] = 0.0
        return tensor


TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(112, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
    GridMask(grid=4, drop_ratio=0.25, p=0.15),
])


class FaceTrainDataset(Dataset):
    def __init__(self, root, parquet_file="test.parquet"):
        self.root = root
        df = pd.read_parquet(os.path.join(root, parquet_file))
        unique_templates = sorted(df["template_id"].unique())
        self.tid_to_label = {tid: i for i, tid in enumerate(unique_templates)}
        self.num_classes = len(unique_templates)
        self.image_paths = df["image_path"].tolist()
        self.labels = [self.tid_to_label[tid] for tid in df["template_id"].values]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.image_paths[idx])
        img = Image.open(path).convert("RGB")
        return TRAIN_TRANSFORM(img), self.labels[idx]


class SubCenterArcFace(nn.Module):
    def __init__(self, embedding_dim, num_classes, K=3, s=64.0, m=0.5):
        super().__init__()
        self.K = K
        self.s = s
        self.m = m
        self.num_classes = num_classes
        self.weight = nn.Parameter(torch.FloatTensor(num_classes * K, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        B = emb.size(0)
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        cosine = cosine.view(B, self.num_classes, self.K).max(dim=2).values
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        return F.cross_entropy(torch.cos(theta + self.m * one_hot) * self.s, labels)

    def sub_center_cosines(self, emb):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        return cosine.view(emb.size(0), self.num_classes, self.K)


class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, s=64.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        return F.cross_entropy(torch.cos(theta + self.m * one_hot) * self.s, labels)


def _make_opt(model, criterion):
    return torch.optim.SGD(
        list(model.parameters()) + list(criterion.parameters()),
        lr=0.1, momentum=0.9, weight_decay=5e-4
    )


def _run_epoch(model, criterion, loader, optimizer, scheduler, device, desc):
    model.train()
    total = 0.0
    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    scheduler.step()
    return total / len(loader)


def run_training(data_root, save_dir, phase, embedding_dim, batch_size, num_workers, device, debug, schedule="step", epochs=35):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    full_ds = FaceTrainDataset(data_root)
    num_classes = full_ds.num_classes

    if debug:
        dataset = Subset(full_ds, list(range(min(500, len(full_ds)))))
        dataset.num_classes = num_classes
        epochs = 2
    else:
        dataset = full_ds

    loader_kwargs = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=(device.type == 'cuda'), drop_last=True)

    flags_path = None
    p1_ckpt = None

    if phase in ("1", "both"):
        model = FaceEncoder(embedding_dim).to(device)
        criterion = SubCenterArcFace(embedding_dim, num_classes).to(device)
        optimizer = _make_opt(model, criterion)
        sched = (
            torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 28, 32], gamma=0.1)
            if schedule == "step" else
            torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
        )
        loader = DataLoader(dataset, shuffle=True, **loader_kwargs)
        for ep in range(epochs):
            avg = _run_epoch(model, criterion, loader, optimizer, sched, device, f"[Ph1 D={embedding_dim}] ep{ep+1}")
        p1_ckpt = save_dir / "phase1.pth"
        torch.save({"model_state_dict": model.state_dict(), "embedding_dim": embedding_dim, "loss": avg, "epoch": epochs-1}, p1_ckpt)

        # noise isolation
        model.eval()
        sub_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        assignments = []
        with torch.inference_mode():
            for imgs, labels in sub_loader:
                cos = criterion.sub_center_cosines(model(imgs.to(device)))
                for i, lab in enumerate(labels.tolist()):
                    assignments.append((lab, cos[i, lab].argmax().item()))
        identity_sub = {}
        for lab, k in assignments:
            identity_sub.setdefault(lab, []).append(k)
        dominant = {lab: Counter(ks).most_common(1)[0][0] for lab, ks in identity_sub.items()}
        noise_flags = np.array([assignments[i][1] != dominant[assignments[i][0]] for i in range(len(assignments))])
        flags_path = save_dir / "noise_flags.npy"
        np.save(flags_path, noise_flags)

    if phase in ("2", "both"):
        if flags_path and os.path.exists(flags_path):
            noise_flags = np.load(flags_path)
            base_ds = FaceTrainDataset(data_root) if not debug else full_ds
            clean_indices = np.where(~noise_flags)[0].tolist()
            if debug:
                clean_indices = [i for i in clean_indices if i < 500]
            clean_dataset = Subset(base_ds, clean_indices)
            clean_dataset.num_classes = num_classes
        else:
            clean_dataset = dataset

        model = FaceEncoder(embedding_dim).to(device)
        if p1_ckpt and os.path.exists(p1_ckpt):
            ckpt = torch.load(p1_ckpt, map_location=device, weights_only=True)
            model.load_state_dict(ckpt["model_state_dict"])
        criterion = ArcFaceLoss(embedding_dim, num_classes).to(device)
        optimizer = _make_opt(model, criterion)
        p2_epochs = 2 if debug else 15
        sched = (
            torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[8, 12, 14], gamma=0.1)
            if schedule == "step" else
            torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
        )
        loader = DataLoader(clean_dataset, shuffle=True, **loader_kwargs)
        for ep in range(p2_epochs):
            avg = _run_epoch(model, criterion, loader, optimizer, sched, device, f"[Ph2 D={embedding_dim}] ep{ep+1}")
        p2_ckpt = save_dir / "phase2.pth"
        torch.save({"model_state_dict": model.state_dict(), "embedding_dim": embedding_dim, "loss": avg, "epoch": p2_epochs-1}, p2_ckpt)
        return str(p2_ckpt)

    return str(p1_ckpt)

## Evaluation Functions

In [ ]:
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

FAR_TARGETS = [1e-4, 1e-5, 1e-6]


class FaceTestDataset(Dataset):
    def __init__(self, root):
        self.root = root
        df = pd.read_parquet(os.path.join(root, "test.parquet"))
        self.image_paths = df["image_path"].tolist()
        self.template_ids = df["template_id"].values
        self.media_ids = df["media_id"].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.image_paths[idx])
        return EVAL_TRANSFORM(Image.open(path).convert("RGB")), idx


def eval_checkpoint(ckpt_path, dataset_root, embedding_dim, batch_size, num_workers, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    dim = ckpt.get("embedding_dim", embedding_dim)
    model = FaceEncoder(dim).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    dataset = FaceTestDataset(dataset_root)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=(device.type == 'cuda'))

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    embs, idxs = [], []
    with torch.inference_mode():
        for imgs, indices in tqdm(loader, desc=f"Eval {Path(ckpt_path).stem}", leave=False):
            embs.append(model.encode(imgs.to(device)).cpu().numpy())
            idxs.append(indices.numpy())
    elapsed = time.perf_counter() - t0
    embeddings = np.vstack(embs)[np.argsort(np.concatenate(idxs))]

    throughput = len(dataset) / elapsed
    peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if device.type == "cuda" else 0.0

    template_features = aggregate_template_features(embeddings, dataset.template_ids, dataset.media_ids)
    pairs_df = pd.read_parquet(os.path.join(dataset_root, "pairs.parquet"))

    tid_list = sorted(template_features.keys())
    tid_to_idx = {tid: i for i, tid in enumerate(tid_list)}
    feat_matrix = np.zeros((len(tid_list), dim), dtype=np.float32)
    for tid, feat in template_features.items():
        feat_matrix[tid_to_idx[tid]] = feat

    t1s = pairs_df["template_id_1"].values
    t2s = pairs_df["template_id_2"].values
    scores = np.zeros(len(pairs_df), dtype=np.float32)
    for i in range(0, len(pairs_df), 500_000):
        end = min(i + 500_000, len(pairs_df))
        scores[i:end] = np.sum(
            feat_matrix[np.array([tid_to_idx[t] for t in t1s[i:end]])] *
            feat_matrix[np.array([tid_to_idx[t] for t in t2s[i:end]])], axis=1
        )

    labels = pairs_df["label"].values
    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]
    neg_sorted = np.sort(neg_scores)[::-1]

    metrics = {}
    for far in FAR_TARGETS:
        idx = max(math.ceil(far * len(neg_scores)) - 1, 0)
        metrics[f"TAR@FAR={far:.0e}"] = float((pos_scores > neg_sorted[idx]).mean()) * 100
    metrics["throughput_img_s"] = throughput
    metrics["peak_gpu_mem_mb"] = peak_mem_mb
    metrics["embedding_dim"] = dim
    metrics["eval_time_s"] = elapsed

    return metrics, scores, labels

## Ablation Grid

In [ ]:
phases = ["1", "2", "both"]
dims   = [128, 256, 512]

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
device = torch.device(DEVICE if (torch.cuda.is_available() or (DEVICE == "mps" and torch.backends.mps.is_available())) else "cpu")
print(f"Device: {device}")
print(f"Grid: {len(phases)} phases × {len(dims)} dims = {len(phases)*len(dims)} runs")

## Run Ablation

In [ ]:
summary_rows = []

for phase, dim in itertools.product(phases, dims):
    tag = f"phase{phase}_d{dim}"
    tag_save_dir = f"{SAVE_DIR}/{tag}"
    print(f"\n{'='*60}")
    print(f"phase={phase}  dim={dim}  ({tag})")
    print("="*60)

    try:
        ckpt_path = run_training(
            DATA_ROOT, tag_save_dir, phase, dim,
            BATCH_SIZE, NUM_WORKERS, device, DEBUG
        )
    except Exception as e:
        print(f"Training failed: {e}")
        continue

    try:
        metrics, _, _ = eval_checkpoint(ckpt_path, DATASET_ROOT, dim, BATCH_SIZE, NUM_WORKERS, device)
    except Exception as e:
        print(f"Eval failed: {e}")
        continue

    row = {**metrics, "phase": phase, "dim": dim, "checkpoint": ckpt_path}
    summary_rows.append(row)
    print(f"  TAR@FAR=1e-4: {metrics.get('TAR@FAR=1e-04', 0):.2f}%  "
          f"TAR@FAR=1e-5: {metrics.get('TAR@FAR=1e-05', 0):.2f}%  "
          f"throughput: {metrics['throughput_img_s']:.1f} img/s")

## Results Summary

In [ ]:
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    cols = ["phase", "dim", "TAR@FAR=1e-04", "TAR@FAR=1e-05", "TAR@FAR=1e-06",
            "throughput_img_s", "peak_gpu_mem_mb", "eval_time_s"]
    cols = [c for c in cols if c in summary_df.columns]

    print("\n" + "="*60)
    print("Ablation Summary")
    print("="*60)
    display(summary_df[cols])

    summary_path = f"{RESULTS_DIR}/summary.csv"
    summary_df[cols].to_csv(summary_path, index=False)
    print(f"\nSaved: {summary_path}")
else:
    print("No results collected.")